# 03 — Pair Eligibility
Filters to H<0.5, computes structural T70 from a common Z0=1.5, and selects the 40 lowest T70 pairs.

In [1]:

import pandas as pd
from src.pair_eligibility import filter_antipersistent_pairs, compute_structural_t70, select_top_pairs_by_structural_t70

In [2]:
fou_parameters = pd.read_parquet("data/processed/fractional_ou_parameters.parquet")
eligible = filter_antipersistent_pairs(fou_parameters)
print("All fOU pairs:", len(fou_parameters))
print("H < 0.5 eligible pairs:", len(eligible))

All fOU pairs: 446
H < 0.5 eligible pairs: 309


In [3]:
STARTING_Z=1.5
TARGET_PROBABILITY=0.70
MAX_HORIZON_DAYS=252
N_PATHS=1000  # switch to 5000 for final run

In [4]:
structural_results = compute_structural_t70(eligible, STARTING_Z, TARGET_PROBABILITY, MAX_HORIZON_DAYS, N_PATHS, seed=42)
print("Reached 70%:", structural_results["structural_t70"].notna().sum())
print("Did not reach 70%:", structural_results["structural_t70"].isna().sum())
structural_results["structural_t70"].describe()

Reached 70%: 262
Did not reach 70%: 47


count    262.000000
mean     148.484733
std       49.567920
min       46.000000
25%      116.000000
50%      142.500000
75%      177.000000
max      252.000000
Name: structural_t70, dtype: float64

In [5]:
selected_pairs = select_top_pairs_by_structural_t70(structural_results, top_n=40)
selected_pairs[["pair","hurst","kappa","sigma","drift_half_life","structural_t70","structural_probability_max"]]

,pair,hurst,kappa,sigma,drift_half_life,structural_t70,structural_probability_max
0,MLM-VMC,0.480033,0.033820,0.014024,20.495399,46.0,0.999
1,SHW-HD,0.479861,0.030470,0.015142,22.748466,50.0,0.999
2,URI-MS,0.466323,0.025619,0.022751,27.055546,58.0,0.997
3,SPGI-MCO,0.476892,0.024497,0.009739,28.295733,62.0,0.997
4,ORCL-ETN,0.480991,0.024537,0.015696,28.248704,62.0,0.995
5,HLT-AXP,0.442941,0.020812,0.017825,33.305885,65.0,0.994
6,V-ROP,0.467707,0.021548,0.014852,32.167630,67.0,0.992
7,PHM-NVR,0.487804,0.022499,0.016728,30.807372,68.0,0.992
8,PEP-HSY,0.452009,0.020260,0.010220,34.213011,70.0,0.992
9,ETN-KLAC,0.495416,0.022089,0.016379,31.380244,71.0,0.990


In [6]:
OUTPUT ="data/processed/eligible_pairs.parquet"
selected_pairs.to_parquet(OUTPUT,index=False)
print(f"Saved {len(selected_pairs)} pairs to {OUTPUT}")

Saved 40 pairs to data/processed/eligible_pairs.parquet
